In [1]:
import os
import requests
import datetime
import pandas as pd

In [10]:
import os
import datetime
import requests
import pandas as pd

API_KEY = os.environ["SOLAR_EDGE_API_KEY"]
SITE_ID = os.environ["SOLAR_EDGE_SITE_ID"]

BASE_URL = f"https://monitoringapi.solaredge.com/site/{SITE_ID}/powerDetails"

START_DATE = datetime.date(2025, 1, 1)
END_DATE = datetime.date(2025, 12, 21)

METERS = "Production,Consumption,SelfConsumption,FeedIn,Purchased"

# =========================
# Helper: Monatsweise Zeiträume erzeugen
# =========================
def month_ranges(start, end):
    current = start
    while current <= end:
        next_month = (current.replace(day=1) + datetime.timedelta(days=32)).replace(day=1)
        month_end = min(next_month - datetime.timedelta(seconds=1), end)
        yield current, month_end
        current = next_month

# =========================
# API-Abfragen & Records sammeln
# =========================
records = []

for start, end in month_ranges(
    START_DATE,
    END_DATE
):
    params = {
        "startTime": start.strftime("%Y-%m-%d %H:%M:%S"),
        "endTime": end.strftime("%Y-%m-%d %H:%M:%S"),
        "meters": METERS,
        "api_key": API_KEY
    }

    print(f"Fetching {params['startTime']} → {params['endTime']}")

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    data = response.json()["powerDetails"]

    for meter in data["meters"]:
        meter_type = meter["type"]
        for v in meter["values"]:
            records.append({
                "date": v["date"],
                "type": meter_type,
                "value": v.get("value", 0)
            })

# =========================
# DataFrame bauen
# =========================
df = pd.DataFrame(records)

df["date"] = pd.to_datetime(df["date"])

df = (
    df.pivot_table(
        index="date",
        columns="type",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
    .sort_values("date")
)

# =========================
# CSV speichern
# =========================
df.to_csv("energy_data.csv", index=False)

print(df.head())
print(df.tail())


Fetching 2025-01-01 00:00:00 → 2025-02-01 00:00:00
Fetching 2025-02-01 00:00:00 → 2025-03-01 00:00:00
Fetching 2025-03-01 00:00:00 → 2025-04-01 00:00:00
Fetching 2025-04-01 00:00:00 → 2025-05-01 00:00:00
Fetching 2025-05-01 00:00:00 → 2025-06-01 00:00:00
Fetching 2025-06-01 00:00:00 → 2025-07-01 00:00:00
Fetching 2025-07-01 00:00:00 → 2025-08-01 00:00:00
Fetching 2025-08-01 00:00:00 → 2025-09-01 00:00:00
Fetching 2025-09-01 00:00:00 → 2025-10-01 00:00:00
Fetching 2025-10-01 00:00:00 → 2025-11-01 00:00:00
Fetching 2025-11-01 00:00:00 → 2025-12-01 00:00:00
Fetching 2025-12-01 00:00:00 → 2025-12-21 00:00:00
type                date  Consumption  FeedIn  Production  Purchased  \
0    2025-01-01 00:00:00    126.95957     0.0         0.0  126.95957   
1    2025-01-01 00:15:00    155.28748     0.0         0.0  155.28748   
2    2025-01-01 00:30:00    128.95338     0.0         0.0  128.95338   
3    2025-01-01 00:45:00    127.86432     0.0         0.0  127.86432   
4    2025-01-01 01:00:00    